# Fourier Analysis of a Periodic Signal

This notebook computes the **complex Fourier coefficients** $C_k$ of a periodic signal,
using **Simpson's rule** for numerical integration.

## Theoretical background

Any periodic signal $f(x)$ with period $T_0$ can be written as a sum of complex sinusoids:

$$f(x) = \sum_{k=-\infty}^{+\infty} C_k \cdot e^{jk\omega_0 x}$$

where $\omega_0 = \frac{2\pi}{T_0}$ is the **fundamental angular frequency**, and the coefficients $C_k$ are given by:

$$C_k = \frac{1}{T_0} \int_0^{T_0} f(x) \cdot e^{-jk\omega_0 x} \, dx$$

## 1. Imports and parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Signal period
period = 2 * np.pi

## 2. Signal definition

We use a simple sinusoidal signal: $f(x) = \sin(x)$

> To test with a different signal, simply modify the `func` function.

In [ ]:
def func(x):
    return np.sin(x)

# Time axis — number of points must be odd (required by Simpson's rule)
x_axis = np.linspace(0, 2 * np.pi, 1001)
y_axis = func(x_axis)

# Plot the signal
plt.figure(figsize=(8, 3))
plt.plot(x_axis, y_axis)
plt.title('Input signal: $f(x) = \sin(x)$')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.grid(True, linestyle='--', color='gray', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Numerical integration — Simpson's rule

Simpson's rule approximates the integral by fitting a **parabola** through each group of 3 consecutive points:

$$\int_a^b f(x)\,dx \approx \frac{h}{3} \sum_{i=0,2,4,...} \left[ y_i + 4y_{i+1} + y_{i+2} \right]$$

where $h$ is the step size between two consecutive points.

> **Constraint**: the number of intervals must be **even** (i.e. the number of points must be **odd**).

In [ ]:
def integration(x, y):
    """Integrate y(x) using the composite Simpson's rule.

    Parameters
    ----------
    x : array  Evenly spaced sample points
    y : array  Signal values (may be complex)

    Returns
    -------
    float or complex : approximate value of the integral
    """
    h = x[1] - x[0]  # constant step size (linspace)
    total = 0
    for i in range(0, len(x) - 2, 2):  # step of 2: process triplets
        total += (h / 3) * (y[i] + 4 * y[i+1] + y[i+2])
    return total

## 4. Computing the Fourier coefficients $C_k$

For each order $k$, we compute:

$$C_k = \frac{1}{T_0} \int_0^{T_0} f(x) \cdot e^{-jk\omega_0 x} \, dx$$

The coefficient $C_0$ (DC component = mean value of the signal) is computed separately.

In [ ]:
def sig_to_coeff(x, y, N):
    """Compute the first N complex Fourier coefficients of signal y(x).

    Returns
    -------
    C0    : order-0 coefficient (DC component)
    Coeff : list of coefficients C1, C2, ..., CN
    """
    w0 = 2 * np.pi / period

    # C0: mean value of the signal (e^0 = 1, so y0 = y)
    C0 = (1 / period) * integration(x, y)

    # Ck for k = 1 to N
    Coeff = []
    for k in range(1, N + 1):
        yk = y * np.exp(-1j * k * w0 * x)  # pointwise product
        Ck = (1 / period) * integration(x, yk)
        Coeff.append(Ck)

    return C0, Coeff


# Compute the first N coefficients
N = 10
C0, FC = sig_to_coeff(x_axis, y_axis, N)

# Build the two-sided spectrum (from -N to +N)
# Hermitian symmetry: C_{-k} = C_k*  =>  |C_{-k}| = |C_k|  and  arg(C_{-k}) = -arg(C_k)
modules   = np.concatenate([np.flip(np.abs(FC)),    [np.abs(C0)],    np.abs(FC)])
arguments = np.concatenate([np.flip(-np.angle(FC)), [np.angle(C0)],  np.angle(FC)])

# Zero out phase for near-zero coefficients (numerical noise)
threshold = 1e-6
arguments[modules < threshold] = 0

## 5. Frequency spectrum

We plot the **amplitude spectrum** $|C_k|$ and the **phase spectrum** $\arg(C_k)$ as a function of order $k$.

For $f(x) = \sin(x)$, the expected result is:
- **Amplitude**: two peaks at $k = \pm 1$ with value $0.5$, zero everywhere else
- **Phase**: $-\pi/2$ at $k = +1$ and $+\pi/2$ at $k = -1$

In [ ]:
k_axis = np.arange(-N, N + 1)  # order axis: from -N to +N

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# --- Amplitude spectrum ---
ax1.axhline(y=0, color='k', linewidth=0.8)
ax1.stem(k_axis, modules)
ax1.set_title('Amplitude spectrum $|C_k|$')
ax1.set_xlabel('k')
ax1.set_ylabel('$|C_k|$')
ax1.grid(True, linestyle='--', color='gray', alpha=0.5)

# --- Phase spectrum ---
ax2.axhline(y=0, color='k', linewidth=0.8)
ax2.stem(k_axis, arguments)
ax2.set_title('Phase spectrum $\\arg(C_k)$')
ax2.set_xlabel('k')
ax2.set_ylabel('$\\arg(C_k)$ [rad]')
ax2.set_ylim(-np.pi - 0.5, np.pi + 0.5)
ax2.grid(True, linestyle='--', color='gray', alpha=0.5)

plt.tight_layout()
plt.show()